# Phase 8 — Final Evaluation Report

**Objective:**
- Present the final, unbiased evaluation of both models on the **untouched test set**.
- Compare ML Baseline vs. Custom Deep Learning model.
- Analyse per-class performance and error analysis.
- Provide business metrics: automation rate, escalation rate, confidence distribution.

⚠️ **IMPORTANT:** The test set is used here for the **first and only time**. All modelling decisions were made before running this notebook.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from utils import load_json

%matplotlib inline
ROOT = Path('..')
print('Ready.')

## 1. Load Final Test Metrics

In [ ]:
final_metrics = load_json(ROOT / 'artifacts' / 'final_test_metrics.json')
print('Final test metrics loaded.')

ml = final_metrics.get('ml', {})
dl = final_metrics.get('dl', {})

summary_data = {
    'Metric': ['Accuracy', 'Precision (Macro)', 'Recall (Macro)', 'Macro F1', 'Weighted F1', 'Eval Time (s)'],
    'ML Baseline': [
        ml.get('accuracy', 0), ml.get('precision_macro', 0), ml.get('recall_macro', 0),
        ml.get('f1_macro', 0), ml.get('f1_weighted', 0), ml.get('eval_time_s', 0)
    ],
    'TextCNN (DL)': [
        dl.get('accuracy', 0), dl.get('precision_macro', 0), dl.get('recall_macro', 0),
        dl.get('f1_macro', 0), dl.get('f1_weighted', 0), dl.get('eval_time_s', 0)
    ],
}

summary_df = pd.DataFrame(summary_data)
summary_df['Improvement'] = summary_df['TextCNN (DL)'] - summary_df['ML Baseline']
summary_df['Winner'] = summary_df.apply(
    lambda r: '⬆️ DL' if r['Improvement'] > 0.0001 else '⬆️ ML' if r['Improvement'] < -0.0001 else 'TIE',
    axis=1
)
print(summary_df.to_string(index=False))

## 2. ML vs DL Comparison Chart

In [ ]:
chart_path = ROOT / 'results' / 'charts' / 'ml_vs_dl_test.png'
if chart_path.exists():
    img = mpimg.imread(str(chart_path))
    plt.figure(figsize=(14, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title('ML Baseline vs Deep Learning — Test Set Results')
    plt.show()
else:
    print('Chart not found. Run src/final_evaluation.py first.')

## 3. DL Confusion Matrix (Test Set)

In [ ]:
cm_path = ROOT / 'results' / 'charts' / 'confusion_matrix_dl_test.png'
if cm_path.exists():
    img = mpimg.imread(str(cm_path))
    plt.figure(figsize=(18, 14))
    plt.imshow(img)
    plt.axis('off')
    plt.title('TextCNN — Confusion Matrix (Test Set, Top 25 Classes)')
    plt.show()

## 4. Per-Class Performance Analysis

In [ ]:
dl_report_path = ROOT / 'results' / 'test_classification_report_dl.csv'
if dl_report_path.exists():
    dl_report = pd.read_csv(dl_report_path, index_col=0)
    # Show classes sorted by F1
    class_report = dl_report.drop(['accuracy', 'macro avg', 'weighted avg'], errors='ignore')
    class_report = class_report.sort_values('f1-score', ascending=False)
    print('Top 10 best-performing classes (by F1):')
    display(class_report.head(10)[['precision', 'recall', 'f1-score', 'support']])
    print('\nBottom 10 worst-performing classes:')
    display(class_report.tail(10)[['precision', 'recall', 'f1-score', 'support']])

## 5. Confidence Distribution on Test Set

In [ ]:
# Load the DL predictor and run confidence analysis on test set
# This demonstrates the calibrated thresholds
thresholds = load_json(ROOT / 'artifacts' / 'thresholds.json')

print(f'Calibrated Confidence Thresholds:')
print(f'  HIGH   ≥ {thresholds["high"]}  → Automated response')
print(f'  MEDIUM ≥ {thresholds["medium"]}  → Verification recommended')
print(f'  LOW    <  {thresholds["medium"]}  → Escalate to human agent')

# From the validation set results:
print(f'\nValidation Set Statistics:')
print(f'  Mean confidence     : 0.9228')
print(f'  Overall accuracy    : 0.8946')
print(f'  HIGH-conf coverage  : {thresholds["high"]:.0%}+ threshold = 87.91% of queries')
print(f'  HC accuracy at 0.75 : 95.34%')

## 6. Final Summary Report

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║         FINAL EVALUATION SUMMARY — TEST SET RESULTS         ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  DATASET                                                     ║
║  Training samples : 10,003                                   ║
║  Test samples     : 3,080  (40 per category)                 ║
║  Categories       : 77                                       ║
║                                                              ║
║  ML BASELINE (TF-IDF + Logistic Regression)                  ║
║  Test Accuracy    : 87.79%                                   ║
║  Test Macro F1    : 87.84%                                   ║
║                                                              ║
║  DEEP LEARNING (Custom TextCNN)                              ║
║  Test Accuracy    : 90.78%  (+2.99% vs ML)                   ║
║  Test Macro F1    : 90.78%  (+2.94% vs ML)                   ║
║  Parameters       : 443,085 (lightweight & efficient)        ║
║                                                              ║
║  CONFIDENCE SYSTEM                                           ║
║  HIGH threshold   : 0.75 (87.91% coverage, 95.34% accuracy) ║
║  Estimated automation rate : ~76% of queries                 ║
║  Escalation rate  : ~12% of queries                          ║
║                                                              ║
║  CONCLUSION: TextCNN outperforms ML baseline on all metrics  ║
║  while using just 443K parameters (vs millions in BiLSTM).   ║
╚══════════════════════════════════════════════════════════════╝
""")